In [ ]:
import os
import math
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType, NumericType, StringType

print("Done")

In [ ]:
BUCKET_NAME = "hcdr"
RAW_PATH     = "s3://{}/raw/".format(BUCKET_NAME)
TRUSTED_PATH = "s3://{}/trusted/".format(BUCKET_NAME)
REFINED_PATH = "s3://{}/refined/eda/".format(BUCKET_NAME)

RAW_DB = "raw_db"
TRUSTED_DB = "hcdr_trusted"

USE_GLUE_CATALOG = True
ALLOW_S3_FALLBACK = False

NULL_ROW_THRESHOLD = 0.7
NULL_COL_THRESHOLD = 0.4

PATHS = {
    "application_train"      : RAW_PATH + "application_train/application_train.csv",
    "bureau"                 : RAW_PATH + "bureau/bureau.csv",
    "bureau_balance"         : RAW_PATH + "bureau_balance/bureau_balance.csv",
    "previous_application"   : RAW_PATH + "previous_application/previous_application.csv",
    "pos_cash_balance"       : RAW_PATH + "pos_cash_balance/POS_CASH_balance.csv",
    "credit_card_balance"    : RAW_PATH + "credit_card_balance/credit_card_balance.csv",
    "installments_payments"  : RAW_PATH + "installments_payments/installments_payments.csv",
}

print("Configuración lista")
print("RAW_PATH     :", RAW_PATH)
print("TRUSTED_PATH :", TRUSTED_PATH)
print("REFINED_PATH :", REFINED_PATH)
print("RAW_DB       :", RAW_DB)
print("USE_GLUE_CATALOG:", USE_GLUE_CATALOG)

In [ ]:
try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext

    sc = SparkContext.getOrCreate()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    print("Spark initialized")
except Exception as e:
    print("ERROR: No connection with GlueContext")
    spark = (
        SparkSession.builder
        .appName("EDA_HomeCreditRisk_SparkSQL_AWS")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.shuffle.partitions", "120")
        .enableHiveSupport()
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## Funciones para limpieza inicial de la data

En esta primera etapa para pasar los datos de `raw` a `trusted` se definio con el equipo las siguientes acciones:

1. Eliminar las columnas con más del 40% de nulos
1. Eliminar las filas con más del 70% de nulos luego de hacer el punto 1 
1. Normalizar el nombre de las columnas, para mejorar el manejo posterior
1. Corregir anomalias, en este caso propias del data set 
1. Para columnas con tipos de datos de string se cambiaron los nulos y los vacions por `UNKNOW` para mejorar el manejo posteior

Se tomo esta desición para evitar datalakage, cualquier imputación estadistica que hagamos en datos de testeo va a llevarnos en incurrir en un error en la metodología puesto que al calcular estos estadisticos, lo hacemos del conjunto de datos enteros, incluyendo el de testeo e introducimos estos datos en el conjunto de entrenamiento.

Tampoco imputamos variables categoricas todavia ya que queremos guardarnos el derecho de tomar desiciones sobre columnas individuales en el `EDA` siguiente.

In [ ]:
def normalize_columns(df):
    for col_name in df.columns:
        new_name = col_name.strip().lower().replace(" ", "_")
        if col_name != new_name:
            df = df.withColumnRenamed(col_name, new_name)
    return df

def fix_anomalies(df):
    anomaly_cols = [
        'days_employed', 'days_first_drawing',
        'days_first_due', 'days_last_due_1st_version',
        'days_last_due', 'days_termination'
    ]
    for col in anomaly_cols:
        if col in df.columns:
            df = df.withColumn(
                col,
                F.when(F.col(col) == 365243, None).otherwise(F.col(col))
            )
    return df


def fill_unknown_strings(df, value="UNKNOW"):
    string_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, StringType)
    ]

    for c in string_cols:
        df = df.withColumn(
            c, F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), value).otherwise(F.col(c))
        )

    return df

def drop_high_null_columns(df, threshold=NULL_COL_THRESHOLD, exclude_cols=None):
    exclude_cols = exclude_cols or []
    total = df.count()
    null_row = df.select([
        F.sum(F.col(c).isNull().cast('int')).alias(c)
        for c in df.columns
    ]).collect()[0]

    to_drop = [
        c for c in df.columns
        if c not in exclude_cols and (null_row[c] / total) >= threshold
    ]

    if to_drop:
        for c in to_drop:
            print('Drop: {} ({:.1f}%)'.format(c, null_row[c] / total * 100))
        df = df.drop(*to_drop)
    else:
        print('No columns dropped')
    return df

def filter_high_null_rows(df, threshold=NULL_ROW_THRESHOLD):
    initial_rows = df.count()
    total_cols = len(df.columns)

    null_count_expr = sum(F.col(c).isNull().cast("int") for c in df.columns)

    df = df.withColumn("row_null_count", null_count_expr)
    df = df.filter((F.col("row_null_count") / total_cols) <= threshold)
    df = df.drop("row_null_count")

    final_rows = df.count()
    rows_dropped = initial_rows - final_rows

    if rows_dropped > 0:
        print("Dropped (> {}% nulls): {}".format(threshold * 100, rows_dropped))
    else:
        print("No rows dropped")

    return df

def process_table(df, table_name, exclude_cols=None, partition_by=None):
    print("--- Processing: {} ---".format(table_name))
    exclude_cols = exclude_cols or []

    df = normalize_columns(df)
    df = fix_anomalies(df)
    df = drop_high_null_columns(df, threshold=NULL_COL_THRESHOLD, exclude_cols=exclude_cols)
    df = filter_high_null_rows(df, threshold=NULL_ROW_THRESHOLD)

    print("Final shape: {} rows x {} cols\n".format(df.count(), len(df.columns)))
    return df

In [ ]:
def load_table(view_name, catalog_table=None, s3_path=None):
    catalog_table = catalog_table or view_name

    if USE_GLUE_CATALOG:
        try:
            full_name = "{}.{}".format(RAW_DB, catalog_table)
            df = spark.table(full_name)
            df = normalize_columns(df)
            df.createOrReplaceTempView(view_name)
            print("SparkSQL created view: {} | {:,} rows * {} columns".format(view_name, df.count(), len(df.columns)))
            return df
        except Exception as e:
            print("\tError:", str(e)[:250])
            if not ALLOW_S3_FALLBACK:
                raise e

    if s3_path is None:
        raise ValueError("No S3 configured route {}".format(view_name))

    df = spark.read.csv(s3_path, header=True, inferSchema=True, nullValue="", nanValue="NaN")
    df = normalize_columns(df)
    df.createOrReplaceTempView(view_name)
    print("SparkSQL created view: {} | {:,} rows * {} columns".format(view_name, df.count(), len(df.columns)))
    return df


def sql(query, rows=20, truncate=False):
    result = spark.sql(query)
    result.show(rows, truncate=truncate)
    return result


def save_on_trusted(df, name, partition_by=None):
    out = TRUSTED_PATH + name + '/'
    writer = df.write \
        .mode('overwrite') \
        .option('compression', 'snappy')
    if partition_by:
        cols = [partition_by] if isinstance(partition_by, str) else partition_by
        writer = writer.partitionBy(*cols)
    writer.parquet(out)
    print('Save strusted/{}'.format(name))


def save_on_refined(df, name):
    out = REFINED_PATH + name + '/'
    df.coalesce(1) \
        .write \
        .mode('overwrite') \
        .option('header', 'true') \
        .csv(out)
    print('Save refined/{}'.format(name))


def build_null_sql(view_name, columns):
    selects = []
    stack_entries = []
    for i, c in enumerate(columns):
        alias = "n_{}".format(i)
        selects.append("SUM(CASE WHEN `{}` IS NULL THEN 1 ELSE 0 END) AS {}".format(c, alias))
        stack_entries.append("'{}', {}".format(c, alias))

    query = """
    WITH base AS (
        SELECT
            COUNT(*) AS total,
            {selects}
        FROM {view_name}
    )
    SELECT
        columna,
        nulos,
        ROUND(100.0 * nulos / total, 2) AS pct_nulos
    FROM base
    LATERAL VIEW stack({n}, {stack_entries}) s AS columna, nulos
    ORDER BY pct_nulos DESC, nulos DESC
    """.format(
        selects=",\n            ".join(selects),
        view_name=view_name,
        n=len(columns),
        stack_entries=", ".join(stack_entries)
    )
    return query


def value_counts_sql(view_name, column_name, target_col="target"):
    return """
    SELECT
        '{col}' AS variable,
        CAST(`{col}` AS STRING) AS valor,
        COUNT(*) AS total,
        SUM(CASE WHEN `{target}` = 1 THEN 1 ELSE 0 END) AS defaults,
        ROUND(100.0 * SUM(CASE WHEN `{target}` = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_pct
    FROM {view}
    WHERE `{col}` IS NOT NULL
      AND `{target}` IN (0, 1)
    GROUP BY `{col}`
    ORDER BY total DESC
    """.format(col=column_name, target=target_col, view=view_name)


def plot_bar_from_pandas(pdf, x, y, title, xlabel=None, ylabel=None, rotation=0):
    if pdf.empty:
        print("\tDataFrame vacio. No se genera grafico.")
        return
    plt.figure(figsize=(10, 4))
    plt.bar(pdf[x].astype(str), pdf[y])
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.xticks(rotation=rotation, ha="right" if rotation else "center")
    plt.tight_layout()
    plt.show()


def plot_hbar_from_pandas(pdf, x, y, title, xlabel=None, ylabel=None):
    if pdf.empty:
        print("\tDataFrame vacio. No se genera grafico.")
        return
    plt.figure(figsize=(10, max(4, len(pdf) * 0.35)))
    plt.barh(pdf[y].astype(str), pdf[x])
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
TABLES = {}

for name, path in PATHS.items():
    TABLES[name] = load_table(name, catalog_table=name, s3_path=path)

app = TABLES["application_train"]
bureau = TABLES["bureau"]
bureau_balance = TABLES["bureau_balance"]
previous_application = TABLES["previous_application"]
pos_cash_balance = TABLES["pos_cash_balance"]
credit_card_balance = TABLES["credit_card_balance"]
installments_payments = TABLES["installments_payments"]

print("\n\tSparkSQL: Tables loaded")
spark.sql("SHOW TABLES").show(truncate=False)

In [ ]:
inventory_rows = []

for nombre, df in TABLES.items():
    temp_view = "tmp_inv_{}".format(nombre)
    spark.sql("""
        SELECT
            '{nombre}' AS tabla,
            COUNT(*) AS filas
        FROM {nombre}
    """.format(nombre=nombre)).createOrReplaceTempView(temp_view)

inventory_query = " UNION ALL ".join([
    "SELECT * FROM tmp_inv_{}".format(nombre) for nombre in TABLES.keys()
])

inventory_sql = spark.sql(inventory_query)
inventory_sql.show(truncate=False)
save_on_trusted(inventory_sql, "inventory_raw")

In [ ]:
schema_rows = []
for nombre, df in TABLES.items():
    for field in df.schema.fields:
        schema_rows.append((nombre, field.name, str(field.dataType)))

schema_df = spark.createDataFrame(schema_rows, ["tabla", "columna", "tipo"])
schema_df.createOrReplaceTempView("schema_inventario")

sql("""
SELECT tabla, columna, tipo
FROM schema_inventario
ORDER BY tabla, columna
""", rows=200, truncate=False)

save_on_refined(schema_df, "schema_inventario")

In [ ]:
nulls_app = spark.sql(build_null_sql("application_train", app.columns))
nulls_app.show(50, truncate=False)

save_on_refined(nulls_app, "nulos_application_train")

nulls_app_pd = nulls_app.limit(30).toPandas()
plot_hbar_from_pandas(
    nulls_app_pd,
    x="pct_nulos",
    y="columna",
    title="Top 30 columnas con mayor porcentaje de nulos — application_train",
    xlabel="% de nulos",
    ylabel="Columna"
)

In [ ]:
exclude_config = {
    "application_train"    : ["sk_id_curr", "target"],
    "bureau"               : ["sk_id_curr", "sk_id_bureau"],
    "bureau_balance"       : ["sk_id_bureau"],
    "previous_application" : ["sk_id_curr", "sk_id_prev"],
    "pos_cash_balance"     : ["sk_id_curr", "sk_id_prev"],
    "credit_card_balance"  : ["sk_id_curr", "sk_id_prev"],
    "installments_payments": ["sk_id_curr", "sk_id_prev"]
}

partition_config = {
    "application_train": "weekday_appr_process_start"
}

trusted_tables = {}

for table_name, raw_df in TABLES.items():
    excluded     = exclude_config.get(table_name, [])
    partition_by = partition_config.get(table_name, None)

    clean_df = process_table(raw_df, table_name, excluded)
    trusted_tables[table_name] = clean_df

    clean_df.createOrReplaceTempView("{}_trusted".format(table_name))
    save_on_trusted(clean_df, table_name, partition_by=partition_by)

print("All tables processed and saved")

### Datos de la tabla principal

Adicionalmente a una versión guardada con las modificaciones anteiormente mencionada decidimos hacer un split, este split puesto que se hace sobre un conjunto de datos desbalanceados, guarda la proporción entre ambos el conjunto de entrenamiento y el de testeo, además el de entrenamiento será imputado con la mediana de ese conjunto de datos, para evitar incurrir en datalakage. Finalmente application fue particionado por la columna 1. Se dejo una versión procesada con los primeros 5 númerales de la tabla principal application, y se crea una version imputada, esta después de hacer un split ***weekday_appr_process_start*** ya que es después del analisis la mejor opción, al estar bastante balanceada esta columna en el dataset.

In [ ]:
spark.sparkContext.setCheckpointDir("s3://hcdr/checkpoints/")
app_trusted = trusted_tables["application_train"]
app_trusted = app_trusted.checkpoint()

seed = 42
train_fraction = 0.8

app_with_split = app_trusted.withColumn("_split_rand", F.rand(seed=seed))

app_train = app_with_split.filter(F.col("_split_rand") < train_fraction).drop("_split_rand")
app_test  = app_with_split.filter(F.col("_split_rand") >= train_fraction).drop("_split_rand")

app_train.cache()
app_train.count()

numeric_cols = [
    f.name for f in app_train.schema.fields
    if isinstance(f.dataType, NumericType) and f.name not in ["target"]
]

median_values = {}
for c in numeric_cols:
    result = app_train.approxQuantile(c, [0.5], 0.001)
    if result and result[0] is not None:
        median_values[c] = result[0]

app_train_imputed = app_train.fillna(median_values)

save_on_trusted(app_test, "application_test", partition_by="weekday_appr_process_start")
save_on_trusted(app_train_imputed, "application_train_imputed", partition_by="weekday_appr_process_start")

In [ ]:
summary_rows = []
for name, df in trusted_tables.items():
    rows      = df.count()
    cols      = len(df.columns)
    num_nulls = sum(
        df.filter(F.col(f.name).isNull()).count()
        for f in df.schema.fields
        if isinstance(f.dataType, NumericType)
    )
    summary_rows.append((name, rows, cols, num_nulls))

summary_df = spark.createDataFrame(
    summary_rows,
    ['table', 'rows', 'cols', 'numeric_nulls']
)
summary_df.show(truncate=False)
save_on_refined(summary_df, 'quality_resume_trusted')